# 👁️ Argos-Action-1.5B v2.0: Fine-Tuning & Held-Out Evaluation Pipeline
**Author:** Pedro Lucas Reis ([@pedroreoli](https://huggingface.co/pedroreoli))

This notebook performs **4-bit QLoRA Fine-Tuning with Assistant-Only Loss Masking** on **Qwen 2.5 1.5B Instruct** using the **Argos Computer Use MCP v2.0 Dataset** (`pedroreoli/argos-computer-use-mcp`), followed by **greedy held-out inference evaluation (`model.generate`)** with 95% Wilson confidence intervals.

### Key v2.0 Methodological Fixes:
1. **Assistant-Only Loss Masking (`labels = -100` on prompt):** Cross-entropy loss is computed strictly on the assistant's JSON output tokens, never on system/AXTree prompt tokens.
2. **Balanced 6-Tool Training & Disjoint Held-Out Split:** All 6 tools (`click`, `type_text`, `select_option`, `scroll`, `press_hotkey`, `escalate_human`) are represented across 1,700 unique training samples and 300 strictly disjoint held-out test samples.
3. **True Post-Training Inference Benchmark:** Evaluates `model.generate()` on unseen prompts rather than static file labels.

In [ ]:
!pip install -q "transformers>=4.45.0" "datasets>=3.0.0" "peft>=0.13.0" "trl>=0.11.0" "bitsandbytes>=0.44.0" accelerate huggingface_hub

In [ ]:
import getpass
from huggingface_hub import login

hf_token = getpass.getpass('Enter your Hugging Face Write Token: ')
login(token=hf_token)

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "pedroreoli/argos-computer-use-mcp"
HUB_MODEL_ID = "pedroreoli/argos-action-1.5b"
MAX_SEQ_LENGTH = 1536

print("Loading Argos v2.0 Dataset...")
dataset = load_dataset(DATASET_ID)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_with_assistant_mask(example: Dict[str, Any]) -> Dict[str, Any]:
    messages = example["messages"]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full_ids = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LENGTH, add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=MAX_SEQ_LENGTH, add_special_tokens=False)["input_ids"]
    prompt_len = min(len(prompt_ids), len(full_ids))
    labels = [-100] * prompt_len + full_ids[prompt_len:]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

@dataclass
class AssistantMaskCollator:
    pad_token_id: int
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids, attention_mask, labels = [], [], []
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.pad_token_id] * pad_len)
            attention_mask.append(f["attention_mask"] + [0] * pad_len)
            labels.append(f["labels"] + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

tokenized_train = dataset["train"].map(tokenize_with_assistant_mask, remove_columns=dataset["train"].column_names)
tokenized_eval = dataset["test"].map(tokenize_with_assistant_mask, remove_columns=dataset["test"].column_names)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="./argos-action-1.5b-v2",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=15,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=AssistantMaskCollator(pad_token_id=tokenizer.pad_token_id)
)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Starting Argos-Action-1.5B v2.0 Training on {gpu_name}...")
train_result = trainer.train()
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

In [ ]:
# Post-Training Held-Out Generation Evaluation (Greedy Decoding on 300 Unseen Test Prompts)
import json, math, time
from collections import Counter

model.eval()
test_rows = dataset["test"]
valid_json, strict_schema, tool_match, esc_total, esc_hits = 0, 0, 0, 0, 0
per_tool_tot, per_tool_hit = Counter(), Counter()

for row in test_rows:
    gold = json.loads(row["output"])
    g_tool = gold["tool_call"]["tool"]
    per_tool_tot[g_tool] += 1
    if g_tool == "escalate_human":
        esc_total += 1
    prompt = tokenizer.apply_chat_template(row["messages"][:-1], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=192, do_sample=False)
    reply = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    try:
        parsed = json.loads(reply)
        valid_json += 1
        if isinstance(parsed, dict) and isinstance(parsed.get("tool_call"), dict):
            p_tool = parsed["tool_call"].get("tool")
            strict_schema += 1
            if p_tool == g_tool:
                tool_match += 1
                per_tool_hit[g_tool] += 1
                if g_tool == "escalate_human":
                    esc_hits += 1
    except Exception:
        pass

n = len(test_rows)
print(f"Held-Out Results (n={n}): Valid JSON={valid_json}/{n} | Strict Schema={strict_schema}/{n} | Tool Match={tool_match}/{n} | Escalation Recall={esc_hits}/{esc_total}")
for t in sorted(per_tool_tot):
    print(f"  - {t}: {per_tool_hit[t]}/{per_tool_tot[t]}")

trainer.push_to_hub(commit_message="feat(model): publish Argos-Action-1.5B v2.0 LoRA policy & evaluation state")
tokenizer.push_to_hub(HUB_MODEL_ID)
print(f"Model published successfully to: https://huggingface.co/{HUB_MODEL_ID}")